## Casual Attention

In [1]:
import torch

# Your Journey starts with one step
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], #x^1
    [0.55, 0.87, 0.66], #x^2
    [0.57, 0.85, 0.64], #x^3
    [0.22, 0.58, 0.33], #x^4
    [0.77, 0.25, 0.10], #x^5
    [0.05, 0.80, 0.55]] #x^6
)


#### Implementation of self attention mechanism with linear layers for query, key and value projections.

##### Arguments:
    * d_in: Input dimension of the feature vectors.
    * d_out: Output dimension of the projected vectors.
    * qkv_bias: Whether to include bias terms in the linear layers.
##### Forward Pass:
    1. Compute keys, queries, and values by passing the input through the respective linear layers.
    2. Calculate attention scores by performing a dot product between queries and keys.
    3. Scale the attention scores by the square root of the key dimension to stabilize gradients.
    4. Apply a causal mask to ensure that each position can only attend to previous positions.
    5. Normalize the masked attention scores using softmax to obtain attention weights.
    6. Compute the context vector as a weighted sum of the values using the attention weights.
    7. Return the context vector as the output of the self-attention mechanism.


In [ ]:
import torch.nn as nn
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in ,d_out, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)

    def forward(self,x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim =-1)
        context_vector = attention_weights @ values
        return context_vector

In [5]:
torch.manual_seed(789)
d_in = inputs.shape[-1]
d_out = 2
sa_v2 = SelfAttention_v2(d_in,d_out)
print(sa_v2(inputs))


tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [6]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attention_scores = queries @ keys.T
print(attention_scores)
attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim =-1)
print(attention_weights)


tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)
tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [11]:
context_length = attention_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [13]:
masked_simple = attention_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [16]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
row_sums

tensor([[0.1921],
        [0.3700],
        [0.5357],
        [0.6775],
        [0.8415],
        [1.0000]], grad_fn=<SumBackward1>)

In [17]:
masked_simple_normalized = masked_simple / row_sums
print(masked_simple_normalized)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


## more Efficient way

In [23]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal = 1)
print(mask)
masked = attention_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [26]:
attention_weights = torch.softmax(masked/keys.shape[-1]**0.5, dim =-1)
print(attention_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [27]:
values = sa_v2.W_value(inputs)
context_vector = attention_weights @ values
print(context_vector)

tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## added dropout

In [32]:
torch.manual_seed(123)
dropout = nn.Dropout(p=0.5)
example= torch.ones(6,6)
print(example)
print(dropout(example))

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [33]:
print(dropout(attention_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4925, 0.4638, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3941, 0.0000],
        [0.3869, 0.3327, 0.0000, 0.3084, 0.3331, 0.3058]],
       grad_fn=<MulBackward0>)


## adding batch support

In [41]:
batch = torch.stack((inputs, inputs), dim = 0)
print(batch.shape)
print(batch.shape[0])

torch.Size([2, 6, 3])
2


In [ ]:
import torch.nn as nn

class CasualAttention(nn.Module):
    '''Implements the casual attention mechanism, 
    which ensures that each token can only attend to previous tokens in the sequence.
    
    Args:
        d_in: The dimensionality of the input token embeddings.
        d_out: The dimensionality of the output token embeddings after attention.
        context_length: The maximum length of the input sequence, used to create the attention mask.
        dropout: The dropout rate applied to the attention weights.
        qkv_bias: Whether to include bias terms in the linear layers for query, key, and value projections.
        register_buffer: A method to register a tensor as a buffer in the module, which is not a parameter but should be part of the module's state.
    
    Forward Pass:
        1. Compute the query, key, and value projections from the input token embeddings.
        2. Calculate the attention scores by performing a dot product between the query and key projections.
        3. Apply the attention mask to prevent attending to future tokens.
        4. Normalize the attention scores using softmax to obtain attention weights.
        5. Apply dropout to the attention weights.
        6. Compute the context vector as a weighted sum of the value projections using the attention weights.
        7. Return the context vector, which contains the attended information for each token in
              the sequence. 
    '''
    def __init__(self, d_in, d_out,context_length, dropout, qkv_bias = False):
        super().__init__()
        self. d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", 
            torch.triu(torch.ones(context_length, context_length), 
            diagonal = 1))


    def forward(self,x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attention_scores = queries @ keys.transpose(1,2)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        # print("attention_scores", masked)
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim =-1)
        attention_weights = self.dropout(attention_weights)
        context_vector = attention_weights @ values
        return context_vector

In [58]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CasualAttention(d_in, d_out,context_length,0.0)
context_vector = ca(batch)
print("context_vector shape:", context_vector.shape)
print(context_vector)



context_vector shape: torch.Size([2, 6, 2])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
